**Chatbot using Hugging Face Transformers**

**Objective**

Build a conversational chatbot using a pre-trained transformer model that generates meaningful, dynamic responses, no hardcoded answers.

In [ ]:
!pip install transformers torch --quiet

print("Libraries installed successfully.")

In [ ]:
# PyTorch — deep learning backend
import torch
# Hugging Face model loaders
from transformers import AutoModelForCausalLM, AutoTokenizer

# for version display
import transformers

print("Imports successful.")
print(f"   PyTorch version      : {torch.__version__}")
print(f"   Transformers version : {transformers.__version__}")


**Model Selection — DialoGPT
GPT-2 **
is trained on general web text (documents, articles).
DialoGPT is fine-tuned specifically for dialogue, making it far better at producing natural conversational replies.






**Pipeline Flow**

User Input  →  Tokenizer (text → token IDs) →  Transformer (multi-head self-attention over token sequence) →  Decoder (auto-regressively generate next tokens) →  Tokenizer (token IDs → text) →  Chatbot Response

**Load Pre-trained Model & Tokenizer**

In [ ]:
#   AutoTokenizer        → converts text <-> token IDs
#   AutoModelForCausalLM → transformer that predicts the next token

MODEL_NAME = "microsoft/DialoGPT-medium"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

model.eval()

total_params = sum(p.numel() for p in model.parameters())
print("Model and tokenizer loaded successfully!")
print(f"Total parameters: {total_params:,}")

**Define the Response Generation Function**

In [ ]:
def generate_response(user_input: str, chat_history_ids=None, max_length: int = 1000):

    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        bot_input_ids = new_input_ids

    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=max_length,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=50,
            top_p=0.95,
            temperature=0.75,
            repetition_penalty=1.3,
        )

    response_tokens = chat_history_ids[:, bot_input_ids.shape[-1]:]
    response_text = tokenizer.decode(
        response_tokens[0],
        skip_special_tokens=True
    )

    return response_text, chat_history_ids


print("generate_response() function defined successfully.")


**Quick Single-Turn Test**

In [ ]:
test_input = "Hello, how are you?"
response, _ = generate_response(test_input)

print(f"User   : {test_input}")
print(f"Chatbot: {response}")

In [ ]:
#Full Interactive Chatbot Loop

def run_chatbot():
    print("=" * 60)
    print("  DialoGPT Chatbot  |  Hugging Face Transformers")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-" * 60)
    print("  Type  exit  or  quit  to end the conversation.")
    print("-" * 60)

    chat_history_ids = None

    while True:
        # accept user input
        user_input = input("You: ").strip()

        # Skip blank / whitespace-only input
        if not user_input:
            print("Chatbot: Please say something. I'm listening!")
            continue

        # Exit condition
        if user_input.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye! It was wonderful talking with you. Have a great day!👋")
            print("=" * 60)
            break

        # Generate response using the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Guard against rare empty-response edge case
        if not response.strip():
            response = "I am not quite sure how to respond to that. Could you rephrase?"

        print(f"Chatbot: {response}")
        print()


run_chatbot()


